# KIT719 Project 1: Information Retrieval System
## Using Reuters Corpus with NLTK

**Group Members:** [Your Name], [Partner Name]
**Student IDs:** [ID1], [ID2]
**Dataset:** Reuters Corpus (NLTK)

**Objective:** build a vector-space (TF-IDF + cosine similarity) and probabilistic (BM25) information retrieval system to retrieve Reuters news articles by topic relevance, and to evaluate and compare these ranking methods, including an advanced query expansion technique.

In [ ]:
import nltk
nltk.download('reuters')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

import string
import math
import re
from collections import defaultdict, Counter

from nltk.corpus import reuters, stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Load & Explore Dataset

In [ ]:
all_files = reuters.fileids()
print('total documents:', len(all_files))
print('total categories:', len(reuters.categories()))

# show first 10 categories
for cat in reuters.categories()[:10]:
    count = len(reuters.fileids(cat))
    print('  ', cat, ':', count, ' documents')

### Select Subset

In [ ]:
selected_categories = ['acq', 'crude', 'trade', 'money-fx', 'interest', 'ship']

doc_ids = []
for cat in selected_categories:
    files = reuters.fileids(cat)
    for f in files:
        if f not in doc_ids:
            doc_ids.append(f)

print('selected documents:', len(doc_ids))

# build document list
docs = []
for doc_id in doc_ids:
    doc_info = {}
    doc_info['id'] = doc_id
    doc_info['text'] = reuters.raw(doc_id)
    doc_info['categories'] = reuters.categories(doc_id)
    docs.append(doc_info)

# CHECK: confirm docs list built correctly
print('docs list length:', len(docs))
print('sample doc id:', docs[0]['id'])
print('sample doc categories:', docs[0]['categories'])
print('sample doc text preview:', docs[0]['text'][:150])

### Text Preprocessing

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """preprocess a single document or query text"""

    # step 1: tokenise
    tokens = word_tokenize(text)

    # step 2: process each token
    processed = []
    for token in tokens:
        word = token.lower()

        # skip punctuation
        if word in string.punctuation:
            continue

        # skip non-alphabetic
        if not word.isalpha():
            continue

        # skip stopwords
        if word in stop_words:
            continue

        # apply stemming
        stemmed = stemmer.stem(word)
        processed.append(stemmed)

    return processed

# CHECK: test preprocessing on a small example before running on full dataset
test_text = "The companies announced a major acquisition and merger deal yesterday."
print('before preprocessing:', test_text)
print('after preprocessing:', preprocess_text(test_text))

# apply preprocessing to all documents and store tokens
for doc_idx in range(len(docs)):
    docs[doc_idx]['tokens'] = preprocess_text(docs[doc_idx]['text'])

# CHECK: confirm tokens were added to every document
print('doc 0 token count:', len(docs[0]['tokens']))
print('doc 0 first 10 tokens:', docs[0]['tokens'][:10])

### Build Inverted Index

In [ ]:
inverted_index = defaultdict(list)
doc_term_freq = []

for doc_idx in range(len(docs)):
    tokens = docs[doc_idx]['tokens']

    # count term frequency
    term_count = {}
    for token in tokens:
        if token not in term_count:
            term_count[token] = 0
        term_count[token] = term_count[token] + 1

    doc_term_freq.append(term_count)

    # add to inverted index
    for token in term_count:
        if doc_idx not in inverted_index[token]:
            inverted_index[token].append(doc_idx)

# CHECK: confirm index and term frequencies built correctly
print('total unique terms in vocabulary:', len(inverted_index))
print('doc_term_freq length (should match number of docs):', len(doc_term_freq))

sample_term = 'compani'  # stemmed form of "company"
if sample_term in inverted_index:
    print(f"'{sample_term}' appears in {len(inverted_index[sample_term])} documents")
    print(f"first 5 doc indices containing it:", inverted_index[sample_term][:5])

### Design Rationale: Inverted Index

A brute-force search would compare a query against all ~3,000 documents on every
request, most of which share no vocabulary with the query at all. The inverted
index restricts candidate retrieval to only documents containing at least one
query term, which is the standard approach in production IR systems (e.g.
Elasticsearch, Lucene) and keeps the system scalable if the corpus size grows.

### Compute TF-IDF

In [ ]:
# compute document frequency
doc_freq = {}
for term in inverted_index:
    doc_freq[term] = len(inverted_index[term])

total_docs = len(docs)

# compute idf
idf = {}
for term in doc_freq:
    df = doc_freq[term]
    idf[term] = math.log(total_docs / df)

# compute tf-idf for each document
doc_vectors = []
for doc_idx in range(total_docs):
    vector = {}
    term_count = doc_term_freq[doc_idx]

    # find max tf for normalisation
    max_tf = 0
    for term in term_count:
        if term_count[term] > max_tf:
            max_tf = term_count[term]

    # compute tf-idf
    for term in term_count:
        tf = term_count[term] / max_tf
        vector[term] = tf * idf[term]

    doc_vectors.append(vector)

# CHECK: confirm idf and vectors computed correctly
print('total documents used for idf:', total_docs)
print(f"idf of '{sample_term}':", idf.get(sample_term, 'not found'))
print('doc_vectors length:', len(doc_vectors))
print('doc 0 vector size (number of unique terms):', len(doc_vectors[0]))
print('doc 0 sample tf-idf values:', dict(list(doc_vectors[0].items())[:5]))

### Cosine Similarity

In [ ]:
def compute_cosine_similarity(vec1, vec2):
    """compute cosine similarity between two vectors"""

    # find common terms
    common_terms = set(vec1.keys()) & set(vec2.keys())

    # compute dot product
    dot_product = 0
    for term in common_terms:
        dot_product = dot_product + vec1[term] * vec2[term]

    # compute magnitude of vec1
    mag1 = 0
    for term in vec1:
        mag1 = mag1 + vec1[term] ** 2
    mag1 = math.sqrt(mag1)

    # compute magnitude of vec2
    mag2 = 0
    for term in vec2:
        mag2 = mag2 + vec2[term] ** 2
    mag2 = math.sqrt(mag2)

    # avoid division by zero
    if mag1 == 0 or mag2 == 0:
        return 0

    return dot_product / (mag1 * mag2)

# CHECK: test cosine similarity with simple known vectors
test_vec1 = {'a': 1, 'b': 2}
test_vec2 = {'a': 1, 'b': 2}  # identical -> should be 1.0
test_vec3 = {'c': 1, 'd': 2}  # no overlap -> should be 0.0
print('identical vectors similarity (expect 1.0):', compute_cosine_similarity(test_vec1, test_vec2))
print('no overlap similarity (expect 0.0):', compute_cosine_similarity(test_vec1, test_vec3))

# CHECK: similarity of a real document with itself should also be 1.0
self_sim = compute_cosine_similarity(doc_vectors[0], doc_vectors[0])
print('doc 0 similarity with itself (expect 1.0):', self_sim)

### Search Function (TF-IDF + Cosine)

In [ ]:
def search_query(query_text, top_k=10):
    """process a user query and return ranked documents"""

    # step 1: preprocess query
    query_tokens = preprocess_text(query_text)

    # step 2: build query vector
    query_term_count = {}
    for token in query_tokens:
        if token not in query_term_count:
            query_term_count[token] = 0
        query_term_count[token] = query_term_count[token] + 1

    # compute tf-idf for query
    query_vector = {}
    max_tf = 0
    for term in query_term_count:
        if query_term_count[term] > max_tf:
            max_tf = query_term_count[term]

    for term in query_term_count:
        if term in idf:
            tf = query_term_count[term] / max_tf
            query_vector[term] = tf * idf[term]

    # step 3: find candidates using inverted index
    candidate_docs = set()
    for token in query_tokens:
        if token in inverted_index:
            for doc_idx in inverted_index[token]:
                candidate_docs.add(doc_idx)

    # step 4: compute similarity
    results = []
    for doc_idx in candidate_docs:
        score = compute_cosine_similarity(query_vector, doc_vectors[doc_idx])
        results.append((doc_idx, score))

    # step 5: sort by score
    results.sort(key=lambda x: x[1], reverse=True)

    return results[:top_k]

# CHECK: run a real query and inspect results
test_query = "company acquisition merger"
results = search_query(test_query, top_k=5)
print(f"query: '{test_query}'")
print('number of candidate docs found:', len(results))
for doc_idx, score in results:
    print(f"  doc {doc_idx} | score {score:.4f} | categories {docs[doc_idx]['categories']}")

### BM25 (Advanced for HD)

In [ ]:
def compute_bm25_score(query_tokens, doc_idx, k1=1.5, b=0.75):
    """compute bm25 score for a document"""

    score = 0

    # compute average document length
    total_length = 0
    for i in range(len(docs)):
        total_length = total_length + len(docs[i]['tokens'])
    avg_doc_len = total_length / len(docs)

    doc_len = len(docs[doc_idx]['tokens'])
    term_count = doc_term_freq[doc_idx]

    for term in query_tokens:
        if term not in term_count:
            continue

        tf = term_count[term]
        df = doc_freq.get(term, 0)

        # idf component
        idf_val = math.log((total_docs - df + 0.5) / (df + 0.5) + 1)

        # tf component with normalisation
        tf_component = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * (doc_len / avg_doc_len)))

        score = score + idf_val * tf_component

    return score

# CHECK: compare bm25 score of a relevant doc vs an unrelated doc
test_tokens = preprocess_text("company acquisition merger")
score_doc0 = compute_bm25_score(test_tokens, 0)
print('bm25 score for doc 0:', score_doc0)
print('avg doc length used:', sum(len(d['tokens']) for d in docs) / len(docs))

### Design Rationale: Why BM25 in Addition to TF-IDF/Cosine

TF-IDF with cosine similarity treats term frequency linearly — a document
mentioning a term 10 times scores roughly twice as relevant as one mentioning
it 5 times, which doesn't match how relevance actually saturates in practice.
BM25 addresses two known weaknesses of the vector-space model:

- **Term frequency saturation** — via the `k1` parameter, additional occurrences
  of a term contribute diminishing returns rather than scaling linearly.
- **Document length normalisation** — via the `b` parameter, BM25 penalises
  long documents that accumulate high term counts simply by being long, rather
  than being genuinely more relevant.

We used the standard default values (`k1=1.5`, `b=0.75`) rather than tuning them,
since parameter tuning was out of scope for this project; this is noted as a
limitation in the evaluation discussion below.

Implementing both methods lets us directly compare a classic vector-space
approach against a probabilistic ranking model on the same corpus and query set.

### BM25 Search Wrapper

In [ ]:
def search_bm25(query_text, top_k=10):
    """process a query using bm25 ranking and return ranked documents"""

    query_tokens = preprocess_text(query_text)

    # find candidates using inverted index
    candidate_docs = set()
    for token in query_tokens:
        if token in inverted_index:
            for doc_idx in inverted_index[token]:
                candidate_docs.add(doc_idx)

    # compute bm25 score for each candidate
    results = []
    for doc_idx in candidate_docs:
        score = compute_bm25_score(query_tokens, doc_idx)
        results.append((doc_idx, score))

    results.sort(key=lambda x: x[1], reverse=True)

    return results[:top_k]

# CHECK: compare bm25 vs tf-idf results side by side for the same query
results_bm25 = search_bm25(test_query, top_k=5)
print(f"BM25 results for '{test_query}':")
for doc_idx, score in results_bm25:
    print(f"  doc {doc_idx} | score {score:.4f} | categories {docs[doc_idx]['categories']}")

### Query Expansion (WordNet Synonyms)

In [ ]:
from nltk.corpus import wordnet

def expand_query(query_tokens, max_synonyms=2):
    """expand query tokens with synonyms found in wordnet"""

    expanded_tokens = []
    for token in query_tokens:
        expanded_tokens.append(token)

        synonyms_found = 0
        for syn in wordnet.synsets(token):
            for lemma in syn.lemmas():
                synonym = lemma.name().lower()
                synonym = synonym.replace('_', ' ')

                # skip multi-word synonyms and duplicates
                if ' ' in synonym:
                    continue
                if synonym == token:
                    continue

                stemmed_synonym = stemmer.stem(synonym)
                if stemmed_synonym not in expanded_tokens:
                    expanded_tokens.append(stemmed_synonym)
                    synonyms_found = synonyms_found + 1

                if synonyms_found >= max_synonyms:
                    break
            if synonyms_found >= max_synonyms:
                break

    return expanded_tokens

# CHECK: see exactly which synonyms got added
original_tokens = preprocess_text("company acquisition merger")
expanded = expand_query(original_tokens)
print('original tokens:', original_tokens)
print('expanded tokens:', expanded)
print('synonyms added:', [t for t in expanded if t not in original_tokens])


def search_query_expanded(query_text, top_k=10):
    """same as search_query but with wordnet query expansion applied first"""

    query_tokens = preprocess_text(query_text)
    query_tokens = expand_query(query_tokens)

    query_term_count = {}
    for token in query_tokens:
        if token not in query_term_count:
            query_term_count[token] = 0
        query_term_count[token] = query_term_count[token] + 1

    query_vector = {}
    max_tf = 0
    for term in query_term_count:
        if query_term_count[term] > max_tf:
            max_tf = query_term_count[term]

    for term in query_term_count:
        if term in idf:
            tf = query_term_count[term] / max_tf
            query_vector[term] = tf * idf[term]

    candidate_docs = set()
    for token in query_tokens:
        if token in inverted_index:
            for doc_idx in inverted_index[token]:
                candidate_docs.add(doc_idx)

    results = []
    for doc_idx in candidate_docs:
        score = compute_cosine_similarity(query_vector, doc_vectors[doc_idx])
        results.append((doc_idx, score))

    results.sort(key=lambda x: x[1], reverse=True)

    return results[:top_k]

# CHECK: compare candidate pool size with vs without expansion
results_expanded = search_query_expanded(test_query, top_k=5)
print('candidates without expansion:', len(search_query(test_query, top_k=1000)))
print('candidates with expansion:', len(results_expanded))

### Design Rationale: Query Expansion via WordNet

Users rarely phrase queries with the exact vocabulary used in the source
documents (e.g. a query for "merger" should ideally also retrieve documents
that only use "acquisition" or "takeover"). WordNet-based synonym expansion
addresses this vocabulary mismatch problem without requiring a hand-built
synonym dictionary.

**Alternative techniques considered but not implemented:**
- *Spelling correction* — not implemented, as the Reuters corpus is professionally
  edited news text with negligible misspellings; the effort would not have
  improved retrieval quality for this dataset. It would matter more for
  user-generated or noisy query logs.
- *Pseudo-relevance feedback (query expansion from top-ranked results)* — a
  stronger technique than static WordNet expansion, but out of scope given
  project time constraints; noted here as a direction for future work.

**Trade-off of the chosen approach:** WordNet expansion is capped at 2 synonyms
per term (`max_synonyms=2`) specifically to limit query drift — expanding too
aggressively risks pulling in documents that share vocabulary but not topic,
which would hurt precision. This trade-off between recall gain and precision
loss is tested directly in the evaluation section below.

### Evaluation Metrics

In [ ]:
def get_relevant_docs(category):
    """treat all documents tagged with this reuters category as relevant"""
    relevant = []
    for doc_idx in range(len(docs)):
        if category in docs[doc_idx]['categories']:
            relevant.append(doc_idx)
    return relevant


def precision_at_k(retrieved_doc_idx, relevant_doc_idx, k):
    top_k_retrieved = retrieved_doc_idx[:k]
    hit_count = 0
    for idx in top_k_retrieved:
        if idx in relevant_doc_idx:
            hit_count = hit_count + 1
    precision = hit_count / k
    return precision


def recall_at_k(retrieved_doc_idx, relevant_doc_idx, k):
    top_k_retrieved = retrieved_doc_idx[:k]
    hit_count = 0
    for idx in top_k_retrieved:
        if idx in relevant_doc_idx:
            hit_count = hit_count + 1
    if len(relevant_doc_idx) == 0:
        return 0
    recall = hit_count / len(relevant_doc_idx)
    return recall


def average_precision(retrieved_doc_idx, relevant_doc_idx, k):
    hit_count = 0
    sum_precision = 0
    for i in range(k):
        doc_idx = retrieved_doc_idx[i]
        if doc_idx in relevant_doc_idx:
            hit_count = hit_count + 1
            precision_at_i = hit_count / (i + 1)
            sum_precision = sum_precision + precision_at_i
    if hit_count == 0:
        return 0
    ap = sum_precision / hit_count
    return ap

# CHECK: test metrics with a made-up example where you know the right answer
fake_retrieved = [1, 2, 3, 4, 5]
fake_relevant = [1, 3, 5, 9, 10]  # 3 of the 5 retrieved are relevant
print('test precision@5 (expect 0.6):', precision_at_k(fake_retrieved, fake_relevant, 5))
print('test recall@5 (expect 0.6):', recall_at_k(fake_retrieved, fake_relevant, 5))
print('test average precision:', average_precision(fake_retrieved, fake_relevant, 5))

# CHECK: real relevant docs for one category
relevant_acq = get_relevant_docs('acq')
print('number of relevant docs for "acq" category:', len(relevant_acq))

### Evaluation Methodology

**Relevance ground truth:** Reuters category tags are used as a proxy for
relevance — a document is treated as relevant to a query if it shares the
query's target category. This is a defensible, reproducible substitute for
manual relevance judgments, but it is an approximation: a document tagged
`acq` is not necessarily relevant to every possible acquisition-related query,
and some genuinely relevant documents may carry a different primary category.
This is a known limitation of category-based evaluation and is revisited below.

**Test query design:** one representative query was constructed per category,
using terms a user might plausibly search with rather than terms copied
directly from the documents, to keep the evaluation realistic rather than
artificially inflating scores.

### Run Evaluation Across Methods

In [ ]:
test_queries = {
    'acq': 'company acquisition merger',
    'crude': 'crude oil petroleum prices',
    'trade': 'trade deficit tariffs',
    'money-fx': 'currency exchange rate',
    'interest': 'interest rate federal reserve',
    'ship': 'shipping vessel cargo'
}

k = 10
methods = ['tfidf_cosine', 'bm25', 'bm25_expanded']

# store precision, recall, ap per method
eval_results = {}
for method in methods:
    eval_results[method] = {'precision': [], 'recall': [], 'ap': []}

for category in test_queries:
    query_text = test_queries[category]
    relevant_doc_idx = get_relevant_docs(category)

    # tf-idf + cosine
    results_tfidf = search_query(query_text, top_k=k)
    retrieved_tfidf = [item[0] for item in results_tfidf]

    # bm25
    results_bm25 = search_bm25(query_text, top_k=k)
    retrieved_bm25 = [item[0] for item in results_bm25]

    # bm25 with expanded query (reuse expand_query, score with bm25)
    expanded_tokens = expand_query(preprocess_text(query_text))
    candidate_docs = set()
    for token in expanded_tokens:
        if token in inverted_index:
            for doc_idx in inverted_index[token]:
                candidate_docs.add(doc_idx)
    scored = []
    for doc_idx in candidate_docs:
        score = compute_bm25_score(expanded_tokens, doc_idx)
        scored.append((doc_idx, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    retrieved_bm25_expanded = [item[0] for item in scored[:k]]

    retrieved_by_method = {
        'tfidf_cosine': retrieved_tfidf,
        'bm25': retrieved_bm25,
        'bm25_expanded': retrieved_bm25_expanded
    }

    # CHECK: print per-query results as we go, so a bad query is caught immediately
    print(f"--- category: {category} | query: '{query_text}' | relevant docs: {len(relevant_doc_idx)} ---")

    for method in methods:
        retrieved = retrieved_by_method[method]
        p = precision_at_k(retrieved, relevant_doc_idx, k)
        r = recall_at_k(retrieved, relevant_doc_idx, k)
        ap = average_precision(retrieved, relevant_doc_idx, k)

        eval_results[method]['precision'].append(p)
        eval_results[method]['recall'].append(r)
        eval_results[method]['ap'].append(ap)

        print(f"  {method}: precision={p:.3f} recall={r:.3f} ap={ap:.3f}")

# build summary table
summary_rows = []
for method in methods:
    avg_precision = sum(eval_results[method]['precision']) / len(eval_results[method]['precision'])
    avg_recall = sum(eval_results[method]['recall']) / len(eval_results[method]['recall'])
    map_score = sum(eval_results[method]['ap']) / len(eval_results[method]['ap'])

    row = {
        'method': method,
        'precision@10': avg_precision,
        'recall@10': avg_recall,
        'MAP': map_score
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print('\n=== FINAL SUMMARY ===')
print(summary_df)

### Visualise Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x_pos = np.arange(len(methods))
bar_width = 0.25

precision_values = summary_df['precision@10'].tolist()
recall_values = summary_df['recall@10'].tolist()
map_values = summary_df['MAP'].tolist()

ax.bar(x_pos - bar_width, precision_values, width=bar_width, label='Precision@10')
ax.bar(x_pos, recall_values, width=bar_width, label='Recall@10')
ax.bar(x_pos + bar_width, map_values, width=bar_width, label='MAP')

ax.set_xticks(x_pos)
ax.set_xticklabels(methods)
ax.set_ylabel('Score')
ax.set_title('Retrieval Method Comparison')
ax.legend()

plt.tight_layout()
plt.show()

### Results Interpretation

**[Fill in actual numbers from your `summary_df` output before submitting —
structure below, replace bracketed values]**

- **TF-IDF + Cosine vs BM25:** BM25 scored [higher/lower/similar] on
  precision@10 ([X] vs [Y]) and MAP ([X] vs [Y]). This is [consistent /
  inconsistent] with the expected behaviour, since BM25's term-frequency
  saturation and length normalisation should reduce the influence of long,
  keyword-heavy documents that TF-IDF may over-rank.

- **BM25 vs BM25 + Query Expansion:** Expansion changed recall@10 from [X] to
  [Y] ([increase/decrease]) and precision@10 from [X] to [Y]. [If recall rose
  and precision fell: this matches the expected trade-off — expansion pulls in
  more loosely related documents, improving coverage at some cost to ranking
  precision.] [If both improved: expansion successfully disambiguated queries
  without introducing significant noise, likely because the category-level
  vocabulary in this corpus is fairly tight.] [If both fell: expansion may have
  introduced synonym drift — WordNet synsets are not domain-specific, and a
  financial-news term like "interest" (rate) can expand toward unrelated senses
  of the word (e.g. "curiosity"), diluting the query.]

- **Limitations affecting these results:** the 6-query evaluation set is small;
  a larger and more diverse query set would give more statistically reliable
  comparisons. The category-based relevance proxy may also under- or
  over-count true relevance for some queries, which could shift the absolute
  metric values without changing the relative ranking between methods.

- **Overall conclusion:** [state which method you'd recommend as the system's
  default, and why — e.g. "BM25 without expansion offers the best
  precision/recall balance for this corpus, while query expansion could be
  revisited with domain-specific synonym filtering as future work."]

### Interactive Search Interface

This cell provides the user-facing entry point required by the project spec: a user
submits a free-text query and receives a ranked list of relevant documents. BM25 is
used as the default ranking method here, since Cell 14/15 showed it performing best
overall (update this justification once real numbers are filled into the Results
Interpretation cell above).

In [ ]:
def run_search_interface():
    """simple command-line interface: user types a query, gets ranked results"""

    print('=== KIT719 IR System ===')
    print('Type a search query, or type quit to exit.')

    while True:
        user_query = input('\nEnter query: ')

        if user_query.lower() == 'quit':
            print('Exiting search.')
            break

        if user_query.strip() == '':
            print('Please enter a non-empty query.')
            continue

        results = search_bm25(user_query, top_k=10)

        if len(results) == 0:
            print('No matching documents found.')
            continue

        print(f"\nTop {len(results)} results for '{user_query}':")
        rank = 1
        for doc_idx, score in results:
            doc_id = docs[doc_idx]['id']
            doc_categories = docs[doc_idx]['categories']
            preview = docs[doc_idx]['text'][:100].replace(chr(10), ' ')
            print(f'{rank}. score={score:.4f} | id={doc_id} | categories={doc_categories}')
            print(f'   preview: {preview}...')
            rank = rank + 1


# run the interactive search loop
run_search_interface()